In [1]:
import pandas as pd
import numpy as np
import os

# ── STEP 1: LOAD DATA ──────────────────────────
# Modify: file path, encoding
df = pd.read_csv(r"C:\Users\vikra\Downloads\HR Analytics Project\HR_Analytics_dirty.csv", 
                 low_memory=False, 
                 encoding='latin1')
print("Original Shape:", df.shape)
print(df.head())

Original Shape: (1510, 38)
    EmpID Age AgeGroup Attrition BusinessTravel  DailyRate  \
0   RM835  34    26-35        No  travel_rarely       1400   
1   RM115  34    26-35        No  Travel_Rarely       1031   
2  RM1064  29    26-35         N  Travel_Rarely       1246   
3  RM1105  29    26-35        No  Travel_Rarely        598   
4   RM235  33    26-35       Yes  Travel_Rarely        813   

                 Department DistanceFromHome Education    EducationField  ...  \
0                       NaN                9         1     Life Sciences  ...   
1  Research and Development                6         4   Life Sciences    ...   
2                     Sales               19         3     Life Sciences  ...   
3    Research & Development                9         3     life sciences  ...   
4    Research & Development               14         3           Medical  ...   

   RelationshipSatisfaction  StandardHours  StockOptionLevel  \
0                         1             80       

In [2]:
#See the Problems First
print("=== MISSING VALUES ===")
print(df.isnull().sum())

print("\n=== DUPLICATE ROWS ===")
print("Duplicates:", df.duplicated().sum())

print("\n=== DATA TYPES ===")
print(df.dtypes)

=== MISSING VALUES ===
EmpID                        0
Age                         41
AgeGroup                     0
Attrition                    0
BusinessTravel              40
DailyRate                    0
Department                  41
DistanceFromHome             0
Education                   47
EducationField               0
EmployeeCount                0
EmployeeNumber               0
EnvironmentSatisfaction      0
Gender                      38
HourlyRate                   0
JobInvolvement               0
JobLevel                     0
JobRole                      0
JobSatisfaction              0
MaritalStatus                0
MonthlyIncome               41
SalarySlab                   0
MonthlyRate                  0
NumCompaniesWorked           0
Over18                       0
OverTime                     0
PercentSalaryHike            0
PerformanceRating            0
RelationshipSatisfaction     0
StandardHours                0
StockOptionLevel             0
TotalWorkingYear

In [37]:
# ── STEP 1: FIX 0 DTYPES ──────────────────────────
int_cols = ['Age', 'Education', 'DistanceFromHome','MonthlyIncome','PercentSalaryHike','YearsAtCompany']
for col in int_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], 
                   errors='coerce').fillna(0).astype(int)
        


In [31]:
# ── STEP 2: FIX 1 - REPLACE NULL STRINGS ───────
# Modify: add more null-like strings if needed
null_values = ['Nill','nill','NILL','NULL','null',
               'N/A','n/a','NA','na','-','none','None']
df.replace(null_values, np.nan, inplace=True)

In [32]:
#---STEP3 : FIX 2 - REMOVE DUPLICATES ───────────────────
before = df.shape[0]
df.drop_duplicates(inplace=True)
print(f"Removed {before - df.shape[0]} duplicates")

Removed 11 duplicates


In [33]:
# ── STEP 3: HANDLE MISSING VALUES ───────────────
# ── STEP 3a: STANDARDIZE MISSING VALUE REPRESENTATIONS ───────────────
# Modify: column names and fill values
missing_tokens = ['NA', 'N/A', 'null', 'NaN', '?', 'missing', '', ' ']

cols_to_check = ['Gender', 'Department', 'BusinessTravel', 'Age', 
                  'Education', 'MonthlyIncome', 'YearsAtCompany']

for col in cols_to_check:
    # strip whitespace first, then replace known missing tokens with actual NaN
    df[col] = df[col].astype(str).str.strip().replace(missing_tokens, np.nan)

# Critical columns — drop the row, no safe way to guess these
df.dropna(subset=['Gender', 'Department', 'BusinessTravel'], inplace=True)

text_fill = {
    'EmpID': 'Unknown',
    'AgeGroup': 'Unknown',
    'Attrition': 'Unknown',
    'BusinessTravel': 'Unknown',
    'Department': 'Unknown',
    'EducationField': 'Unknown',
    'Gender': 'Unknown',
    'JobRole': 'Unknown',
    'MaritalStatus': 'Unknown',
    'SalarySlab': 'Unknown',
    'Over18': 'Unknown',
    'OverTime': 'Unknown',
}
numeric_fill = {
    'Age': 0,
    'DailyRate': 0,
    'DistanceFromHome': 0,
    'Education': 0,
    'EmployeeCount': 0,
    'EmployeeNumber': 0,
    'EnvironmentSatisfaction': 0,
    'JobInvolvement': 0,
    'JobLevel': 0,
    'JobSatisfaction': 0,
    'MonthlyIncome': 0,
    'MonthlyRate': 0,
    'NumCompaniesWorked': 0,
    'PercentSalaryHike': 0,
    'PerformanceRating': 0,
    'RelationshipSatisfaction': 0,
    'StandardHours': 0,
    'StockOptionLevel': 0,
    'TotalWorkingYears': 0,
    'TrainingTimesLastYear': 0,
    'WorkLifeBalance': 0,
    'YearsAtCompany': 0,
    'YearsInCurrentRole': 0,
    'YearsSinceLastPromotion': 0,
    'YearsWithCurrManager': 0,
    
}
df.fillna(text_fill, inplace=True)
df.fillna(numeric_fill, inplace=True)

print("Missing values after fix:")
print(df.isnull().sum())

Missing values after fix:
EmpID                       0
Age                         0
AgeGroup                    0
Attrition                   0
BusinessTravel              0
DailyRate                   0
Department                  0
DistanceFromHome            0
Education                   0
EducationField              0
EmployeeCount               0
EmployeeNumber              0
EnvironmentSatisfaction     0
Gender                      0
HourlyRate                  0
JobInvolvement              0
JobLevel                    0
JobRole                     0
JobSatisfaction             0
MaritalStatus               0
MonthlyIncome               0
SalarySlab                  0
MonthlyRate                 0
NumCompaniesWorked          0
Over18                      0
OverTime                    0
PercentSalaryHike           0
PerformanceRating           0
RelationshipSatisfaction    0
StandardHours               0
StockOptionLevel            0
TotalWorkingYears           0
TrainingTimesL

In [34]:
#Fix 4: Fix Case Inconsistency & Whitespace


text_cols = ['EmpID', 'AgeGroup', 'Attrition', 'BusinessTravel', 'Department',
             'EducationField', 'Gender', 'JobRole', 'MaritalStatus', 'SalarySlab',
             'Over18', 'OverTime']

# Step 1: strip whitespace + fix casing for ALL text columns
for col in text_cols:
    df[col] = df[col].astype(str).str.strip().str.title()

# Step 2: convert title-cased missing tokens to real NaN, across ALL text columns at once
missing_tokens = ['Nan', 'Na', 'N/A', 'Null', '?', 'Missing', 'None', '']
df[text_cols] = df[text_cols].replace(missing_tokens, np.nan)

# Step 3: column-specific synonym/abbreviation mapping
# Add an entry here ONLY for columns where you actually find inconsistent variants
column_mappings = {
    'Gender': {
        'F': 'Female', 'M': 'Male'
    },
    'OverTime': {
        'True': 'Yes', 'Y': 'Yes', '1': 'Yes',
        'False': 'No', 'N': 'No', '0': 'No'
    },
    'Attrition': {
        'True': 'Yes', 'Y': 'Yes', '1': 'Yes',
        'False': 'No', 'N': 'No', '0': 'No'
    },
    'Department': {
        'Hr': 'Human Resources', 'Human Resource': 'Human Resources',
        'R&D': 'Research & Development', 'Research And Development': 'Research & Development',
        'Saless': 'Sales', 'Sales Dept': 'Sales'
    }
}

for col, mapping in column_mappings.items():
    df[col] = df[col].replace(mapping)

# Step 4: now check EVERY text column, not just three, to confirm nothing's left messy
for col in text_cols:
    print(f"--- {col} ---")
    print(df[col].value_counts(dropna=False))
    print()

--- EmpID ---
EmpID
Rm1464    2
Rm1466    2
Rm1470    2
Rm1467    2
Rm1461    2
         ..
Rm1304    1
Rm344     1
Rm1242    1
Rm1465    1
Rm1213    1
Name: count, Length: 1235, dtype: int64

--- AgeGroup ---
AgeGroup
26-35    514
36-45    402
46-55    183
18-25    104
55+       37
Name: count, dtype: int64

--- Attrition ---
Attrition
No     1044
Yes     196
Name: count, dtype: int64

--- BusinessTravel ---
BusinessTravel
Travel_Rarely        849
Travel_Frequently    228
Non-Travel           123
NaN                   32
Travelrarely           8
Name: count, dtype: int64

--- Department ---
Department
Research & Development    796
Sales                     356
Human Resources            50
NaN                        38
Name: count, dtype: int64

--- EducationField ---
EducationField
Life Sciences       512
Medical             395
Marketing           130
Technical Degree    112
Other                67
Human Resources      24
Name: count, dtype: int64

--- Gender ---
Gender
Male      72

In [39]:
#Fix 5: Remove Negative Outliers + Unrealistic Outliers
before = df.shape[0]

# Define realistic bounds based on domain knowledge (not just >= 0)
df = df[(df['Age'] >= 18) & (df['Age'] <= 65)]
df = df[(df['MonthlyIncome'] >= 1000) & (df['MonthlyIncome'] <= 200000)]
df = df[(df['YearsAtCompany'] >= 0) & (df['YearsAtCompany'] <= 40)]

after = df.shape[0]
print(f"Removed {before - after} rows with negative/unrealistic values")

Removed 0 rows with negative/unrealistic values


In [38]:
# ── STEP 6: FINAL VERIFICATION ─────────────────
print("\n=== FINAL SHAPE ===")
print(df.shape)
print("\n=== REMAINING NULLS ===")
print(df.isnull().sum())
print("\n=== DTYPES ===")
print(df.dtypes)


=== FINAL SHAPE ===
(1240, 38)

=== REMAINING NULLS ===
EmpID                        0
Age                          0
AgeGroup                     0
Attrition                    0
BusinessTravel              32
DailyRate                    0
Department                  38
DistanceFromHome             0
Education                    0
EducationField               0
EmployeeCount                0
EmployeeNumber               0
EnvironmentSatisfaction      0
Gender                      31
HourlyRate                   0
JobInvolvement               0
JobLevel                     0
JobRole                      0
JobSatisfaction              0
MaritalStatus                0
MonthlyIncome                0
SalarySlab                   0
MonthlyRate                  0
NumCompaniesWorked           0
Over18                       0
OverTime                     0
PercentSalaryHike            0
PerformanceRating            0
RelationshipSatisfaction     0
StandardHours                0
StockOptionLe

In [40]:
# ── STEP 11: SAVE CLEANED FILE ──────────────────
# Modify: output path and filename
os.makedirs(r"C:\Users\vikra\Downloads\HR Analytics Project\cleaned", exist_ok=True)
df.to_csv(r"C:\Users\vikra\Downloads\HR Analytics Project\cleaned\HR_Analytics_CLEAN.csv", index=False)
print("\n✅ File cleaned and saved!")


✅ File cleaned and saved!


In [26]:
#Generate a Profile Report on Cleaned Data (Best Way)
from data_profiling import ProfileReport

profile = ProfileReport(df, title="HR Analytics CLEANED Profile", minimal=True)
profile.to_file(r"C:\Users\vikra\Downloads\HR Analytics Project\reports\HR_Analytics_CLEANED_profile.html")
print("Clean report saved!")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|█████████████████████████████████████████████████████████████████████████████████| 38/38 [00:00<00:00, 538.90it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

Clean report saved!
